# Data Preparation and Feature Engineering

**Project:** Used Heavy Equipment Price Prediction and Market Intelligence  
**Source:** Blue Book for Bulldozers — historical `Train.csv` (1989–2011)

This notebook documents the deterministic data-quality rules and feature engineering used in the original modelling experiments. It preserves raw columns, retains incomplete transactions, and prepares five predictors used in the Random Forest and CatBoost comparisons. The notebook does **not** train a predictive model or claim new validation performance.

**Setup:** Place `Train.csv` in `data/raw/`. Open Jupyter from the repository root or `notebooks/` directory. The accompanying `src/preprocessing.py` module must be present.

## 1. Environment and data source

Loading fails explicitly when the source file cannot be found. The project-relative location is portable across operating systems.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

# Support Jupyter launched either from the repository root or notebooks/.
workdir = Path.cwd().resolve()
if (workdir / "src" / "preprocessing.py").exists():
    project_root = workdir
elif (workdir.parent / "src" / "preprocessing.py").exists():
    project_root = workdir.parent
else:
    raise FileNotFoundError("Open Jupyter from the repository root or its notebooks/ folder.")

sys.path.insert(0, str(project_root / "src"))
from preprocessing import clean_transactions, catboost_features, PREDICTORS

input_path = project_root / "data" / "raw" / "Train.csv"
if not input_path.exists():
    raise FileNotFoundError(f"Place the Kaggle dataset at: {input_path}")

raw = pd.read_csv(input_path, low_memory=False, parse_dates=["saledate"])
print(f"Raw dataset: {len(raw):,} transactions × {raw.shape[1]} columns")
print("Date range:", raw["saledate"].min().date(), "to", raw["saledate"].max().date())

Raw dataset: 401,125 transactions × 53 columns
Date range: 1989-01-17 to 2011-12-30


## 2. Data-quality assessment

Missing values are not equivalent to invalid values. For example, `YearMade` contains the sentinel value 1000, which must be checked separately from `NaN`. The operating-hours variable also includes genuine zeros, missing values, and implausibly large positive values.

In [2]:
# Show completeness of the fields used to build model predictors.
source_cols = ["SalePrice", "saledate", "YearMade", "MachineHoursCurrentMeter", "ProductGroup", "ModelID", "ProductSize"]
source_audit = pd.DataFrame({
    "MissingCount": raw[source_cols].isna().sum(),
    "MissingPercent": raw[source_cols].isna().mean() * 100,
})
display(source_audit.round(2))
print("Manufacturing-year sentinel (1000):", int(raw["YearMade"].eq(1000).sum()))

,MissingCount,MissingPercent
SalePrice,0,0.00
saledate,0,0.00
YearMade,0,0.00
MachineHoursCurrentMeter,258360,64.41
ProductGroup,0,0.00
ModelID,0,0.00
ProductSize,210775,52.55


Manufacturing-year sentinel (1000): 38185


## 3. Reproducible cleaning and derived predictors

The `clean_transactions` function implements the rules recovered from the original analysis:

- Set `YearMade == 1000` to missing; also set manufacturing years later than the sale year to missing.
- Calculate `EquipmentAge = SaleYear − YearMade_Clean`.
- Flag operating-hour readings above `(EquipmentAge + 1) × 365.25 × 24`; set flagged readings to missing in `MachineHours_Clean`.
- Preserve original columns, distinguishing recorded zeros from missing hours.

The operating-hour threshold is a conservative *physical feasibility screen*, not a claim about realistic annual machine utilization. When age is missing, this rule cannot assess the operating-hours reading.

In [3]:
cleaned = clean_transactions(raw)
assert len(cleaned) == len(raw), "Cleaning must not drop transactions."
assert cleaned["EquipmentAge"].dropna().ge(0).all(), "Negative equipment ages remain."
assert cleaned.loc[cleaned["Hours_PhysicalRule_Flag"].eq(1), "MachineHours_Clean"].isna().all()

quality_summary = pd.Series({
    "Transactions retained": len(cleaned),
    "Manufacturing-year values invalidated": int(cleaned["YearMade_Invalid_Flag"].sum()),
    "Equipment ages unavailable": int(cleaned["EquipmentAge"].isna().sum()),
    "Operating-hours values physically inconsistent": int(cleaned["Hours_PhysicalRule_Flag"].sum()),
    "Operating-hours missing after cleaning": int(cleaned["MachineHours_Clean"].isna().sum()),
    "Recorded zero operating-hours values": int(raw["MachineHoursCurrentMeter"].eq(0).sum()),
}, name="Count")
display(quality_summary.to_frame())

,Count
Transactions retained,401125
Manufacturing-year values invalidated,38198
Equipment ages unavailable,38198
Operating-hours values physically inconsistent,378
Operating-hours missing after cleaning,258738
Recorded zero operating-hours values,73126


## 4. Cleaning verification

The original analysis found **38,198 unavailable equipment ages** after applying the manufacturing-year checks. This is a useful regression check, not a modelling metric. The raw records are retained, and flagged values remain auditable through the new indicator columns.

In [4]:
expected_rows = 401_125
expected_missing_age = 38_198
assert len(cleaned) == expected_rows, "Unexpected dataset size; check the input CSV."
assert cleaned["EquipmentAge"].isna().sum() == expected_missing_age, (
    "Age-cleaning counts differ from the original project; inspect input or rules."
)

columns_to_check = ["YearMade", "YearMade_Clean", "SaleYear", "EquipmentAge", "MachineHoursCurrentMeter", "MachineHours_Clean", "Hours_PhysicalRule_Flag"]
display(cleaned[columns_to_check].head(10))

,YearMade,YearMade_Clean,SaleYear,EquipmentAge,MachineHoursCurrentMeter,MachineHours_Clean,Hours_PhysicalRule_Flag
0,2004,2004.0,2006,2.0,68.0,68.0,0
1,1996,1996.0,2004,8.0,4640.0,4640.0,0
2,2001,2001.0,2004,3.0,2838.0,2838.0,0
3,2001,2001.0,2011,10.0,3486.0,3486.0,0
4,2007,2007.0,2009,2.0,722.0,722.0,0
5,2004,2004.0,2008,4.0,508.0,508.0,0
6,1993,1993.0,2004,11.0,11540.0,11540.0,0
7,2001,2001.0,2005,4.0,4883.0,4883.0,0
8,2008,2008.0,2009,1.0,302.0,302.0,0
9,1000,NaN,2007,NaN,20700.0,20700.0,0


## 5. Model-ready features

The five predictors reflect the final experimental models. `ModelID` is an identifier and is treated as a **categorical** feature, not a continuous quantity. `ProductGroup` and `ProductSize` are also categorical. Missing categorical values become the explicit string `"Missing"`; missing numerical values remain `NaN` for CatBoost. Different algorithms may need algorithm-specific preprocessing fitted using **training data only**.

In [5]:
X = catboost_features(cleaned)
y = cleaned["SalePrice"].copy()

assert X.columns.tolist() == PREDICTORS
assert len(X) == len(y) == len(raw)
assert not X[["ProductGroup", "ModelID", "ProductSize"]].isna().any().any()

print("Model-ready feature types:")
print(X.dtypes)
print("\nMissing predictor values:")
display(X.isna().sum().to_frame("MissingCount"))
display(X.head())

Model-ready feature types:
EquipmentAge          float64
MachineHours_Clean    float64
ProductGroup              str
ModelID                   str
ProductSize               str
dtype: object

Missing predictor values:


,MissingCount
EquipmentAge,38198
MachineHours_Clean,258738
ProductGroup,0
ModelID,0
ProductSize,0


,EquipmentAge,MachineHours_Clean,ProductGroup,ModelID,ProductSize
0,2.0,68.0,WL,3157,Missing
1,8.0,4640.0,WL,77,Medium
2,3.0,2838.0,SSL,7009,Missing
3,10.0,3486.0,TEX,332,Small
4,2.0,722.0,SSL,17311,Missing


## 6. Chronological development protocol

The original modelling study compared rolling five-year training windows against the subsequent two years: **2001–2005 → 2006–2007**, **2003–2007 → 2008–2009**, and **2005–2009 → 2010–2011**. These windows were repeatedly examined during development, so they are **development validation periods**, not a fully independent prospective test. No validation targets are used to engineer features here.

In [6]:
periods = [
    ("A", 2001, 2005, 2006, 2007),
    ("B", 2003, 2007, 2008, 2009),
    ("C", 2005, 2009, 2010, 2011),
]
rows = []
for name, start, end, vstart, vend in periods:
    train = cleaned.loc[cleaned["SaleYear"].between(start, end)]
    valid = cleaned.loc[cleaned["SaleYear"].between(vstart, vend)]
    assert train["saledate"].max() < valid["saledate"].min()
    rows.append({"Experiment": name, "TrainingYears": f"{start}–{end}",
                 "ValidationYears": f"{vstart}–{vend}",
                 "TrainingRows": len(train), "ValidationRows": len(valid)})
display(pd.DataFrame(rows))

,Experiment,TrainingYears,ValidationYears,TrainingRows,ValidationRows
0,A,2001–2005,2006–2007,90436,53893
1,B,2003–2007,2008–2009,109489,83616
2,C,2005–2009,2010–2011,157972,68587


## 7. Data export and hand-off

The processed dataset is intentionally **not** exported by default: it is large and can be regenerated from `Train.csv` using `src/preprocessing.py`. A later modelling notebook can call `clean_transactions(raw)` and `catboost_features(cleaned)` directly. If a local cache is useful, run the optional export below. Git excludes `data/processed/` from version control.

In [7]:
# Optional local export; leave False for normal repository use.
EXPORT_PROCESSED_CSV = False
if EXPORT_PROCESSED_CSV:
    destination = project_root / "data" / "processed" / "train_prepared.csv"
    destination.parent.mkdir(parents=True, exist_ok=True)
    cleaned.to_csv(destination, index=False)
    print("Saved:", destination)
else:
    print("No processed CSV exported. Cleaning remains reproducible from the raw dataset.")

No processed CSV exported. Cleaning remains reproducible from the raw dataset.


## Conclusions and limitations

The preprocessing workflow preserves transaction coverage while correcting two identified data-quality problems: placeholder/inconsistent manufacturing years and operating-hour measurements exceeding an age-based physical bound. The resulting age and hours features preserve missingness rather than converting unknown values to zero. Category-aware missingness remains important, particularly because `ProductSize` is absent for entire equipment groups.

**Limitations:** The feasibility rule for operating hours does not detect every erroneous measurement; records with unknown age cannot be evaluated by that rule. This notebook does not prove the selected five features are optimal, nor does it establish future predictive performance. Subsequent modelling must distinguish development validation from an independent test.